# CUPED Comparison: Previous vs Next Period

**Objectives:** Compare outcomes between a previous period and a next period using CUPED variance reduction. The workflow builds the CUPED baseline from the previous period only, applies the adjustment to both periods, and analyzes per-case effects with bootstrap uncertainty quantification (bootstrap CIs reported per `TestCaseId`).

In [13]:
# Cell 1 — Configuration & Library Imports
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict
from IPython.display import display

try:
    from statsmodels.api import OLS, add_constant
except ModuleNotFoundError:
    OLS = None
    add_constant = None

# Plot settings
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Configuration
PREVIOUS_PATH = Path('data/JetStream_Baseline.csv')
NEXT_PATH = Path('data/JetStream_PostBaseline.csv')
N_BOOT = 10_000
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Configuration loaded:')
print(f"Previous path: {PREVIOUS_PATH}")
print(f"Next path: {NEXT_PATH}")
print(f"Bootstrap iterations: {N_BOOT}")
print(f"Random seed: {RANDOM_SEED}")

Configuration loaded:
Previous path: data/JetStream_Baseline.csv
Next path: data/JetStream_PostBaseline.csv
Bootstrap iterations: 10000
Random seed: 42


In [14]:
# Cell 2 — Robust File Loaders
REQUIRED_COLUMNS = ['RunId', 'TrialId', 'TestCaseId', 'LLMScore', 'Value']

def load_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")
    if path.suffix.lower() == '.csv':
        df = pd.read_csv(path)
    elif path.suffix.lower() in {'.parquet', '.pq'}:
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported file type for {path}. Use CSV or Parquet.")
    missing_cols = set(REQUIRED_COLUMNS) - set(df.columns)
    if missing_cols:
        raise ValueError(f"Dataset {path} is missing required columns: {sorted(missing_cols)}")
    # Drop TrialID explicitly as instructed (single TrialID only).
    df = df.drop(columns=['TrialId'])
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
    if df['Value'].isna().any():
        bad_rows = df[df['Value'].isna()]
        raise ValueError(
            f"Non-numeric values detected in 'Value' column for dataset {path}. "
            f"Offending row indices (first 5): {bad_rows.index.tolist()[:5]}"
        )
    return df

previous_df = load_dataset(PREVIOUS_PATH)
next_df = load_dataset(NEXT_PATH)
print('Loaded previous:', previous_df.shape)
print('Loaded next:', next_df.shape)


Loaded previous: (109, 4)
Loaded next: (60, 4)


In [15]:
# Cell 3 — Data Hygiene & Coverage Snapshot

def coverage_snapshot(prev: pd.DataFrame, nxt: pd.DataFrame) -> pd.DataFrame:
    summary_rows = []
    metrics = sorted(set(prev['LLMScore']).union(nxt['LLMScore']))
    for metric in metrics:
        prev_cases = set(prev.loc[prev['LLMScore'] == metric, 'TestCaseId'])
        next_cases = set(nxt.loc[nxt['LLMScore'] == metric, 'TestCaseId'])
        inter = prev_cases & next_cases
        coverage = len(inter) / len(next_cases) if next_cases else np.nan
        summary_rows.append({
            'LLMScore': metric,
            'cases_prev': len(prev_cases),
            'cases_next': len(next_cases),
            'cases_intersection': len(inter),
            'coverage_rate_next': coverage,
            'low_overlap_flag': (not np.isnan(coverage)) and (coverage < 0.7)
        })
    return pd.DataFrame(summary_rows)

prev_counts = previous_df.groupby('LLMScore').agg(
    rows=('TestCaseId', 'size'),
    unique_cases=('TestCaseId', pd.Series.nunique)
).reset_index()
next_counts = next_df.groupby('LLMScore').agg(
    rows=('TestCaseId', 'size'),
    unique_cases=('TestCaseId', pd.Series.nunique)
).reset_index()
print('Previous snapshot:')
display(prev_counts)
print('Next snapshot:')
display(next_counts)
coverage_df = coverage_snapshot(previous_df, next_df)
print('Coverage overview:')
display(coverage_df)

Previous snapshot:


,LLMScore,rows,unique_cases
0,AIFoundry/tool_call_accuracy,109,13


Next snapshot:


,LLMScore,rows,unique_cases
0,AIFoundry/tool_call_accuracy,60,8


Coverage overview:


,LLMScore,cases_prev,cases_next,cases_intersection,coverage_rate_next,low_overlap_flag
0,AIFoundry/tool_call_accuracy,13,8,8,1.0,False


In [16]:
# Cell 4 — Build CUPED Baseline (X) from Previous
X_baseline = (
    previous_df
    .groupby(['TestCaseId', 'LLMScore'], as_index=False)['Value']
    .mean()
    .rename(columns={'Value': 'X_baseline'})
)
metric_baseline_stats = (
    X_baseline.groupby('LLMScore')['X_baseline']
    .agg(['mean', 'std', 'count'])
    .rename(columns={'count': 'n_cases'})
    .reset_index()
)
print('Baseline preview:')
display(X_baseline.head())
print('Baseline statistics per metric:')
display(metric_baseline_stats)

Baseline preview:


,TestCaseId,LLMScore,X_baseline
0,AddKnowledge,AIFoundry/tool_call_accuracy,4.000000
1,CallConfigurationSubAgentV2,AIFoundry/tool_call_accuracy,3.260870
2,CallExecutionSubAgentV2,AIFoundry/tool_call_accuracy,3.500000
3,CallKnowledgeSubAgent,AIFoundry/tool_call_accuracy,3.250000
4,ConfigureAgent,AIFoundry/tool_call_accuracy,3.615385


Baseline statistics per metric:


,LLMScore,mean,std,n_cases
0,AIFoundry/tool_call_accuracy,3.249805,1.189197,13


In [17]:
# Cell 5 — Estimate θ per Metric (CUPED coefficient)
prev_with_baseline = previous_df.merge(X_baseline, on=['TestCaseId', 'LLMScore'], how='left')
theta_rows = []
for metric, group in prev_with_baseline.groupby('LLMScore'):
    x = group['X_baseline']
    y = group['Value']
    x_centered = x - x.mean()
    y_centered = y - y.mean()
    var_x = x_centered.var(ddof=0)
    if np.isclose(var_x, 0):
        theta = 0.0
        warn = True
    else:
        cov = np.mean(x_centered * y_centered)
        theta = cov / var_x
        warn = False
    corr_xy = np.corrcoef(x, y)[0, 1] if len(group) > 1 else np.nan
    ols_slope = np.nan
    if OLS is not None and not np.isclose(var_x, 0):
        X = add_constant(x_centered)
        model = OLS(y_centered, X).fit()
        ols_slope = model.params[1]
    theta_rows.append({
        'LLMScore': metric,
        'theta': theta,
        'corr_XY': corr_xy,
        'var_X': var_x,
        'N_rows_prev': len(group),
        'OLS_slope': ols_slope,
        'variance_or_corr_flag': warn or (not np.isnan(corr_xy) and abs(corr_xy) < 0.05)
    })

theta_df = pd.DataFrame(theta_rows)
print('Theta estimates per metric:')
display(theta_df)

Theta estimates per metric:


/tmp/ipykernel_24581/3313215851.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ols_slope = model.params[1]


,LLMScore,theta,corr_XY,var_X,N_rows_prev,OLS_slope,variance_or_corr_flag
0,AIFoundry/tool_call_accuracy,1.0,0.425887,0.61902,109,1.0,False


In [18]:
# Cell 6 — Apply CUPED to Previous & Next
mean_X_baseline = X_baseline.groupby('LLMScore')['X_baseline'].mean().rename('mean_X_baseline')
X_with_mean = X_baseline.merge(mean_X_baseline, on='LLMScore', how='left')

def apply_cuped(df: pd.DataFrame, label: str) -> pd.DataFrame:
    merged = df.merge(X_with_mean, on=['TestCaseId', 'LLMScore'], how='left')
    merged = merged.merge(theta_df[['LLMScore', 'theta']], on='LLMScore', how='left')
    merged['X_baseline'] = merged['X_baseline'].fillna(merged['mean_X_baseline'])
    merged['theta'] = merged['theta'].fillna(0.0)
    merged['Value_CUPED'] = merged['Value'] - merged['theta'] * (merged['X_baseline'] - merged['mean_X_baseline'])
    merged['Period'] = label
    return merged

previous_cuped = apply_cuped(previous_df, 'Previous')
next_cuped = apply_cuped(next_df, 'Next')

variance_rows = []
for metric, group in pd.concat([previous_cuped, next_cuped]).groupby('LLMScore'):
    var_raw = group['Value'].var(ddof=0)
    var_cuped = group['Value_CUPED'].var(ddof=0)
    reduction = 1 - (var_cuped / var_raw) if var_raw > 0 else np.nan
    variance_rows.append({
        'LLMScore': metric,
        'var_raw': var_raw,
        'var_cuped': var_cuped,
        'variance_reduction_pct': reduction * 100 if not np.isnan(reduction) else np.nan
    })
variance_df = pd.DataFrame(variance_rows)
print('Variance diagnostics per metric:')
display(variance_df)

Variance diagnostics per metric:


,LLMScore,var_raw,var_cuped,variance_reduction_pct
0,AIFoundry/tool_call_accuracy,3.418158,2.931293,14.243473


In [19]:
# Cell 7 — Per-Case & Per-Metric Comparisons (CUPED space)
all_cuped = pd.concat([previous_cuped, next_cuped], ignore_index=True)
per_case = (
    all_cuped
    .groupby(['TestCaseId', 'LLMScore', 'Period'])['Value_CUPED']
    .mean()
    .unstack('Period')
    .rename(columns={'Previous': 'Prev_CUPED_Mean', 'Next': 'Next_CUPED_Mean'})
)
per_case['Diff'] = per_case['Next_CUPED_Mean'] - per_case['Prev_CUPED_Mean']
per_case = per_case.reset_index()
print('Per-case CUPED summary:')
display(per_case.head())

def cohen_d(diff_values: np.ndarray) -> float:
    diff_values = diff_values[~np.isnan(diff_values)]
    if diff_values.size == 0:
        return np.nan
    mean_diff = diff_values.mean()
    std_diff = diff_values.std(ddof=1)
    return mean_diff / std_diff if std_diff > 0 else np.nan

metric_summary_rows = []
for metric, group in per_case.groupby('LLMScore'):
    diffs = group['Diff'].dropna().values
    prev_mean = group['Prev_CUPED_Mean'].mean()
    next_mean = group['Next_CUPED_Mean'].mean()
    percent_change = (next_mean / prev_mean - 1) if prev_mean != 0 else np.nan
    share_improved = np.mean(diffs > 0) if len(diffs) else np.nan
    metric_summary_rows.append({
        'LLMScore': metric,
        'mean_diff': np.mean(diffs) if len(diffs) else np.nan,
        'cohens_d': cohen_d(diffs),
        'percent_change': percent_change,
        'share_improved': share_improved,
        'n_cases': len(diffs)
    })
metric_summary_df = pd.DataFrame(metric_summary_rows)
print('Per-metric CUPED summary:')
display(metric_summary_df)

Per-case CUPED summary:


Period,TestCaseId,LLMScore,Next_CUPED_Mean,Prev_CUPED_Mean,Diff
0,AddKnowledge,AIFoundry/tool_call_accuracy,4.249805,3.249805,1.000000
1,CallConfigurationSubAgentV2,AIFoundry/tool_call_accuracy,3.988936,3.249805,0.739130
2,CallExecutionSubAgentV2,AIFoundry/tool_call_accuracy,3.049805,3.249805,-0.200000
3,CallKnowledgeSubAgent,AIFoundry/tool_call_accuracy,4.999805,3.249805,1.750000
4,ConfigureAgent,AIFoundry/tool_call_accuracy,3.634420,3.249805,0.384615


Per-metric CUPED summary:


,LLMScore,mean_diff,cohens_d,percent_change,share_improved,n_cases
0,AIFoundry/tool_call_accuracy,0.619067,0.945635,0.190493,0.75,8


In [20]:
# Cell 8 — Bootstrap Inference (decomposed per TestCaseId; optional per-metric within case)
from numpy.random import default_rng

rng = default_rng(RANDOM_SEED)

def bootstrap_two_sample_mean_diff(prev_vals: np.ndarray, next_vals: np.ndarray, n_boot: int) -> np.ndarray:
    prev_vals = np.asarray(prev_vals, dtype=float)
    next_vals = np.asarray(next_vals, dtype=float)
    if prev_vals.size == 0 or next_vals.size == 0:
        return np.array([])
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        prev_s = rng.choice(prev_vals, size=prev_vals.size, replace=True)
        next_s = rng.choice(next_vals, size=next_vals.size, replace=True)
        boot[i] = next_s.mean() - prev_s.mean()
    return boot

# Aggregate per TestCaseId (across metrics) — used for the uplift bar chart
bootstrap_rows = []
bootstrap_results: Dict[str, np.ndarray] = {}

for case_id, case_group in all_cuped.groupby('TestCaseId'):
    prev_vals = (
        case_group.loc[case_group['Period'] == 'Previous']
        .groupby('RunId')['Value_CUPED']
        .mean()
        .dropna()
        .values
    )
    next_vals = (
        case_group.loc[case_group['Period'] == 'Next']
        .groupby('RunId')['Value_CUPED']
        .mean()
        .dropna()
        .values
    )
    if prev_vals.size == 0 or next_vals.size == 0:
        continue

    point_est = next_vals.mean() - prev_vals.mean()
    boot_diffs = bootstrap_two_sample_mean_diff(prev_vals, next_vals, N_BOOT)
    if boot_diffs.size == 0:
        continue

    bootstrap_results[str(case_id)] = boot_diffs
    se = boot_diffs.std(ddof=1)
    ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])

    bootstrap_rows.append({
        'TestCaseId': case_id,
        'mean_diff': point_est,
        'SE': se,
        'CI_lo_95': ci_lo,
        'CI_hi_95': ci_hi,
        'N_prev_runs': int(prev_vals.size),
        'N_next_runs': int(next_vals.size),
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)
if not bootstrap_df.empty:
    bootstrap_df = bootstrap_df.sort_values('mean_diff', ascending=False).reset_index(drop=True)
    print('Bootstrap summary per test case (top 20 by mean diff):')
    display(bootstrap_df.head(20))
    print(f"Computed per-test-case bootstrap for {len(bootstrap_df)} test cases.")
else:
    print('Bootstrap summary per test case: (no overlapping test cases with data in both periods)')

# Decomposed per TestCaseId × LLMScore (metric within case)
bootstrap_case_metric_rows = []
bootstrap_case_metric_results: Dict[tuple, np.ndarray] = {}

for (case_id, metric), group in all_cuped.groupby(['TestCaseId', 'LLMScore']):
    prev_vals = (
        group.loc[group['Period'] == 'Previous']
        .groupby('RunId')['Value_CUPED']
        .mean()
        .dropna()
        .values
    )
    next_vals = (
        group.loc[group['Period'] == 'Next']
        .groupby('RunId')['Value_CUPED']
        .mean()
        .dropna()
        .values
    )
    if prev_vals.size == 0 or next_vals.size == 0:
        continue

    point_est = next_vals.mean() - prev_vals.mean()
    boot_diffs = bootstrap_two_sample_mean_diff(prev_vals, next_vals, N_BOOT)
    if boot_diffs.size == 0:
        continue

    bootstrap_case_metric_results[(case_id, metric)] = boot_diffs
    se = boot_diffs.std(ddof=1)
    ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])
    bootstrap_case_metric_rows.append({
        'TestCaseId': case_id,
        'LLMScore': metric,
        'mean_diff': point_est,
        'SE': se,
        'CI_lo_95': ci_lo,
        'CI_hi_95': ci_hi,
        'N_prev_runs': int(prev_vals.size),
        'N_next_runs': int(next_vals.size),
    })

bootstrap_case_metric_df = pd.DataFrame(bootstrap_case_metric_rows)
if not bootstrap_case_metric_df.empty:
    bootstrap_case_metric_df = bootstrap_case_metric_df.sort_values(['TestCaseId', 'LLMScore']).reset_index(drop=True)
    print('Bootstrap summary per test case × metric (sample 25 rows):')
    display(bootstrap_case_metric_df.head(25))
else:
    print('Bootstrap summary per test case × metric: (no overlapping data)')

Bootstrap summary per test case (top 20 by mean diff):


,TestCaseId,mean_diff,SE,CI_lo_95,CI_hi_95,N_prev_runs,N_next_runs
0,CallKnowledgeSubAgent,1.750000,0.637267,0.625000,3.000000,8,2
1,AddKnowledge,1.000000,0.468258,0.000000,2.000000,3,1
2,configure_federated_knowledge,1.000000,0.707483,0.000000,2.000000,2,1
3,CallConfigurationSubAgentV2,0.739130,0.723099,-0.714286,2.043634,23,7
4,postMessageCanary,0.478788,0.469318,-0.444848,1.389121,33,25
5,ConfigureAgent,0.384615,0.981511,-1.692308,2.076923,13,4
6,CallExecutionSubAgentV2,-0.200000,0.755813,-1.650000,1.275000,8,10
7,ExecuteTask,-0.200000,0.771077,-1.725000,1.275000,8,10


Computed per-test-case bootstrap for 8 test cases.
Bootstrap summary per test case × metric (sample 25 rows):


,TestCaseId,LLMScore,mean_diff,SE,CI_lo_95,CI_hi_95,N_prev_runs,N_next_runs
0,AddKnowledge,AIFoundry/tool_call_accuracy,1.000000,0.471543,0.000000,2.000000,3,1
1,CallConfigurationSubAgentV2,AIFoundry/tool_call_accuracy,0.739130,0.719825,-0.763975,2.043478,23,7
2,CallExecutionSubAgentV2,AIFoundry/tool_call_accuracy,-0.200000,0.765330,-1.675000,1.275000,8,10
3,CallKnowledgeSubAgent,AIFoundry/tool_call_accuracy,1.750000,0.637044,0.625000,3.000000,8,2
4,ConfigureAgent,AIFoundry/tool_call_accuracy,0.384615,0.998780,-1.692308,2.076923,13,4
5,ExecuteTask,AIFoundry/tool_call_accuracy,-0.200000,0.750236,-1.700000,1.250000,8,10
6,configure_federated_knowledge,AIFoundry/tool_call_accuracy,1.000000,0.701601,0.000000,2.000000,2,1
7,postMessageCanary,AIFoundry/tool_call_accuracy,0.478788,0.467402,-0.448485,1.396424,33,25


In [21]:
# Cell 9 — Visuals: Distribution & Effect Views
plot_dir = Path('outputs/plots')
plot_dir.mkdir(parents=True, exist_ok=True)

def _safe_filename(s: str) -> str:
    # Keep filenames stable across OS/filesystems
    return ''.join(ch if ch.isalnum() or ch in ('-', '_', '.') else '_' for ch in str(s))

def plot_metric_kdes(metric: str):
    metric_data = all_cuped[all_cuped['LLMScore'] == metric]
    plt.figure()
    sns.kdeplot(data=metric_data, x='Value_CUPED', hue='Period', common_norm=False)
    plt.title(f'CUPED Value Distribution — {metric}')
    plt.xlabel('Value (CUPED)')
    plt.ylabel('Density')
    plt.tight_layout()
    path = plot_dir / f"kde_cuped_{_safe_filename(metric)}.png"
    plt.savefig(path)
    plt.close()
    return path

def plot_test_case_kde(case_id):
    # Aggregate across metrics by RunId within each period so each run contributes once per period
    case_group = all_cuped[all_cuped['TestCaseId'] == case_id]
    if case_group.empty:
        return None
    case_runs = (
        case_group
        .groupby(['Period', 'RunId'], as_index=False)['Value_CUPED']
        .mean()
    )
    if case_runs.empty:
        return None
    plt.figure()
    sns.kdeplot(data=case_runs, x='Value_CUPED', hue='Period', common_norm=False)
    plt.title(f'CUPED Value Distribution — TestCaseId={case_id}')
    plt.xlabel('Value (CUPED)')
    plt.ylabel('Density')
    plt.tight_layout()
    case_dir = plot_dir / 'kde_cuped_by_test_case'
    case_dir.mkdir(parents=True, exist_ok=True)
    path = case_dir / f"kde_cuped_case_{_safe_filename(case_id)}.png"
    plt.savefig(path)
    plt.close()
    return path

def plot_bootstrap_kde_test_case(case_id, boot_diffs: np.ndarray):
    boot_diffs = np.asarray(boot_diffs, dtype=float)
    if boot_diffs.size == 0:
        return None
    plt.figure()
    sns.kdeplot(boot_diffs, fill=True)
    plt.axvline(0, color='black', linestyle='--', label='Zero')
    plt.axvline(np.mean(boot_diffs), color='blue', linestyle='-', label='Mean')
    ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])
    plt.axvline(ci_lo, color='red', linestyle='--', label='95% CI')
    plt.axvline(ci_hi, color='red', linestyle='--')
    plt.title(f'Bootstrap Mean Diff — TestCaseId={case_id}')
    plt.xlabel('Bootstrap Mean Diff (Next - Previous)')
    plt.ylabel('Density')
    plt.legend()
    plt.tight_layout()
    case_dir = plot_dir / 'kde_bootstrap_by_test_case'
    case_dir.mkdir(parents=True, exist_ok=True)
    path = case_dir / f"kde_bootstrap_case_{_safe_filename(case_id)}.png"
    plt.savefig(path)
    plt.close()
    return path

def plot_bootstrap_kde_test_case_metric(case_id, metric: str, boot_diffs: np.ndarray):
    boot_diffs = np.asarray(boot_diffs, dtype=float)
    if boot_diffs.size == 0:
        return None
    plt.figure()
    sns.kdeplot(boot_diffs, fill=True)
    plt.axvline(0, color='black', linestyle='--', label='Zero')
    plt.axvline(np.mean(boot_diffs), color='blue', linestyle='-', label='Mean')
    ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])
    plt.axvline(ci_lo, color='red', linestyle='--', label='95% CI')
    plt.axvline(ci_hi, color='red', linestyle='--')
    plt.title(f'Bootstrap Mean Diff — TestCaseId={case_id} — {metric}')
    plt.xlabel('Bootstrap Mean Diff (Next - Previous)')
    plt.ylabel('Density')
    plt.legend()
    plt.tight_layout()
    case_dir = plot_dir / 'kde_bootstrap_by_test_case_metric'
    case_dir.mkdir(parents=True, exist_ok=True)
    path = case_dir / f"kde_bootstrap_case_{_safe_filename(case_id)}__metric_{_safe_filename(metric)}.png"
    plt.savefig(path)
    plt.close()
    return path

metric_plot_paths = {}
for metric in all_cuped['LLMScore'].unique():
    metric_plot_paths[metric] = {'cuped_kde': plot_metric_kdes(metric)}

# Save CUPED distributions per test case id
case_plot_paths = {}
for case_id in sorted(all_cuped['TestCaseId'].unique()):
    p = plot_test_case_kde(case_id)
    if p is not None:
        case_plot_paths[case_id] = p

# Save bootstrap KDEs per test case (aggregate across metrics)
bootstrap_case_kde_paths = {}
if 'bootstrap_results' in globals():
    for case_id_str, boot_diffs in bootstrap_results.items():
        p = plot_bootstrap_kde_test_case(case_id_str, boot_diffs)
        if p is not None:
            bootstrap_case_kde_paths[case_id_str] = p

# Save bootstrap KDEs per test case × metric (decomposed)
bootstrap_case_metric_kde_paths = {}
if 'bootstrap_case_metric_results' in globals():
    for (case_id, metric), boot_diffs in bootstrap_case_metric_results.items():
        p = plot_bootstrap_kde_test_case_metric(case_id, metric, boot_diffs)
        if p is not None:
            bootstrap_case_metric_kde_paths[(case_id, metric)] = p

bar_plot_path = None
if bootstrap_df is not None and not bootstrap_df.empty:
    fig_w = max(10, 0.35 * max(1, len(bootstrap_df)))
    fig, ax = plt.subplots(figsize=(fig_w, 6))
    bar_plot = sns.barplot(data=bootstrap_df, x='TestCaseId', y='mean_diff', palette='viridis', ax=ax)
    positions = ax.get_xticks()
    ax.errorbar(
        positions, bootstrap_df['mean_diff'],
        yerr=[bootstrap_df['mean_diff'] - bootstrap_df['CI_lo_95'], bootstrap_df['CI_hi_95'] - bootstrap_df['mean_diff']],
        fmt='none', c='black', capsize=3
    )
    ax.axhline(0, color='black', linestyle='--')
    ax.set_title('Mean CUPED Difference per Test Case with 95% CI')
    ax.set_xlabel('TestCaseId')
    ax.set_ylabel('Mean Diff (Next - Previous)')
    for tick in ax.get_xticklabels():
        tick.set_rotation(90)
        tick.set_horizontalalignment('center')
    fig.tight_layout()
    bar_plot_path = plot_dir / 'mean_diff_bar_by_test_case.png'
    fig.savefig(bar_plot_path)
    plt.close(fig)
else:
    print('Skipping uplift bar chart (bootstrap_df is empty).')

print('Plot files saved:')
for metric, paths in metric_plot_paths.items():
    for kind, path in paths.items():
        print(metric, kind, path)
print(f"Saved {len(case_plot_paths)} test-case CUPED distribution plots under: {plot_dir / 'kde_cuped_by_test_case'}")
print(f"Saved {len(bootstrap_case_kde_paths)} test-case bootstrap KDE plots under: {plot_dir / 'kde_bootstrap_by_test_case'}")
print(f"Saved {len(bootstrap_case_metric_kde_paths)} test-case×metric bootstrap KDE plots under: {plot_dir / 'kde_bootstrap_by_test_case_metric'}")
print('Bar chart path:', bar_plot_path)

/tmp/ipykernel_24581/3842979794.py:35: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(data=case_runs, x='Value_CUPED', hue='Period', common_norm=False)
/tmp/ipykernel_24581/3842979794.py:35: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(data=case_runs, x='Value_CUPED', hue='Period', common_norm=False)
/tmp/ipykernel_24581/3842979794.py:35: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(data=case_runs, x='Value_CUPED', hue='Period', common_norm=False)
/tmp/ipykernel_24581/3842979794.py:35: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(data=case_runs, x='Value_CUPED', hue='Period', common_norm=False)
/tmp/ipykernel_24581/3842979794.py:35: UserWarning: Dataset 

Plot files saved:
AIFoundry/tool_call_accuracy cuped_kde outputs/plots/kde_cuped_AIFoundry_tool_call_accuracy.png
Saved 13 test-case CUPED distribution plots under: outputs/plots/kde_cuped_by_test_case
Saved 8 test-case bootstrap KDE plots under: outputs/plots/kde_bootstrap_by_test_case
Saved 8 test-case×metric bootstrap KDE plots under: outputs/plots/kde_bootstrap_by_test_case_metric
Bar chart path: outputs/plots/mean_diff_bar_by_test_case.png


/tmp/ipykernel_24581/3842979794.py:124: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  bar_plot = sns.barplot(data=bootstrap_df, x='TestCaseId', y='mean_diff', palette='viridis', ax=ax)


In [22]:
# Cell 10 — Multiple-Comparison Context (optional; per TestCaseId)
try:
    from statsmodels.stats.multitest import multipletests
    from scipy.stats import ttest_ind
except ImportError:
    multipletests = None
    ttest_ind = None

if ttest_ind is not None and bootstrap_df is not None and not bootstrap_df.empty:
    pvals = []
    for case_id in bootstrap_df['TestCaseId']:
        case_group = all_cuped[all_cuped['TestCaseId'] == case_id]
        prev_vals = (
            case_group.loc[case_group['Period'] == 'Previous']
            .groupby('RunId')['Value_CUPED']
            .mean()
            .dropna()
            .values
        )
        next_vals = (
            case_group.loc[case_group['Period'] == 'Next']
            .groupby('RunId')['Value_CUPED']
            .mean()
            .dropna()
            .values
        )
        if prev_vals.size < 2 or next_vals.size < 2:
            pvals.append(np.nan)
            continue
        pvals.append(ttest_ind(next_vals, prev_vals, equal_var=False).pvalue)

    temp_df = bootstrap_df.copy()
    temp_df['p_value_welch_ttest'] = pvals
    valid_mask = temp_df['p_value_welch_ttest'].notna()
    if multipletests is not None and valid_mask.any():
        _, q_values, _, _ = multipletests(temp_df.loc[valid_mask, 'p_value_welch_ttest'], method='fdr_bh')
        temp_df.loc[valid_mask, 'q_value_bh'] = q_values
    print('Multiple comparison table (exploratory; across test cases):')
    display(temp_df.head(50))
else:
    print('Multiple comparison analysis skipped (dependencies unavailable or insufficient data).')

Multiple comparison table (exploratory; across test cases):


/home/aadkannan/venvs/singlenotebooks/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


,TestCaseId,mean_diff,SE,CI_lo_95,CI_hi_95,N_prev_runs,N_next_runs,p_value_welch_ttest,q_value_bh
0,CallKnowledgeSubAgent,1.750000,0.637267,0.625000,3.000000,8,2,0.035770,0.214617
1,AddKnowledge,1.000000,0.468258,0.000000,2.000000,3,1,NaN,NaN
2,configure_federated_knowledge,1.000000,0.707483,0.000000,2.000000,2,1,NaN,NaN
3,CallConfigurationSubAgentV2,0.739130,0.723099,-0.714286,2.043634,23,7,0.354552,0.709103
4,postMessageCanary,0.478788,0.469318,-0.444848,1.389121,33,25,0.319530,0.709103
5,ConfigureAgent,0.384615,0.981511,-1.692308,2.076923,13,4,0.746953,0.808118
6,CallExecutionSubAgentV2,-0.200000,0.755813,-1.650000,1.275000,8,10,0.808118,0.808118
7,ExecuteTask,-0.200000,0.771077,-1.725000,1.275000,8,10,0.808118,0.808118


In [23]:
# Cell 11 — Sensitivity Analyses
intersection_cases = set(previous_df['TestCaseId']).intersection(set(next_df['TestCaseId']))
per_case_intersection = per_case[per_case['TestCaseId'].isin(intersection_cases)].copy()
intersection_summary = (
    per_case_intersection.groupby('LLMScore')['Diff']
    .agg(mean_diff_intersection='mean', n_cases_intersection='count')
)

def winsorize(series: pd.Series, lower=0.025, upper=0.975) -> pd.Series:
    if series.isna().all():
        return series
    lower_bound = series.quantile(lower)
    upper_bound = series.quantile(upper)
    return series.clip(lower_bound, upper_bound)

per_case_winsor = per_case.copy()
per_case_winsor['Diff_winsor'] = per_case.groupby('LLMScore')['Diff'].transform(winsorize)
winsor_summary = (
    per_case_winsor.groupby('LLMScore')['Diff_winsor']
    .agg(mean_diff_winsor='mean')
)
median_summary = (
    per_case.groupby('LLMScore')['Diff']
    .agg(median_diff='median')
)

sensitivity_df = (
    metric_summary_df[['LLMScore', 'mean_diff', 'n_cases']]
    .merge(intersection_summary, on='LLMScore', how='left')
    .merge(winsor_summary, on='LLMScore', how='left')
    .merge(median_summary, on='LLMScore', how='left')
)
print('Sensitivity analyses overview:')
display(sensitivity_df)

Sensitivity analyses overview:


,LLMScore,mean_diff,n_cases,mean_diff_intersection,n_cases_intersection,mean_diff_winsor,median_diff
0,AIFoundry/tool_call_accuracy,0.619067,8,0.619067,8,0.60266,0.608959


In [24]:
# Cell 12 — Final Summary Tables & Exports
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

per_case_path = output_dir / 'per_case_cuped_diff.csv'
metric_summary_path = output_dir / 'metric_cuped_summary_raw.csv'
bootstrap_path = output_dir / 'test_case_cuped_summary_bootstrap.csv'
bootstrap_case_metric_path = output_dir / 'test_case_metric_cuped_summary_bootstrap.csv'
theta_path = output_dir / 'theta_table.csv'
variance_path = output_dir / 'variance_reduction_by_metric.csv'
coverage_path = output_dir / 'coverage_by_metric.csv'

per_case.to_csv(per_case_path, index=False)
metric_summary_df.to_csv(metric_summary_path, index=False)
bootstrap_df.to_csv(bootstrap_path, index=False)
bootstrap_case_metric_df.to_csv(bootstrap_case_metric_path, index=False)
theta_df.to_csv(theta_path, index=False)
variance_df.to_csv(variance_path, index=False)
coverage_df.to_csv(coverage_path, index=False)

print('Exports saved:')
for path in [per_case_path, metric_summary_path, bootstrap_path, bootstrap_case_metric_path, theta_path, variance_path, coverage_path]:
    print(path)
print('Plots directory:', plot_dir)

Exports saved:
outputs/per_case_cuped_diff.csv
outputs/metric_cuped_summary_raw.csv
outputs/test_case_cuped_summary_bootstrap.csv
outputs/test_case_metric_cuped_summary_bootstrap.csv
outputs/theta_table.csv
outputs/variance_reduction_by_metric.csv
outputs/coverage_by_metric.csv
Plots directory: outputs/plots


## Interpretation Guide

* **Primary unit:** The primary reported bootstrap uncertainty is per **test case** (`TestCaseId`), using resampling over runs within each period.
* **Effect size:** Positive mean differences indicate higher CUPED-adjusted values in Next vs Previous for that test case.
* **Uncertainty:** 95% bootstrap confidence intervals that exclude zero indicate stronger evidence that the test case effect is non-zero.
* **Variance reduction:** Compare raw vs CUPED variance (still reported per `LLMScore`); larger reductions imply a more informative baseline (`corr(X, Y)` close to ±1).
* **Coverage:** Low overlap between previous and next test cases weakens comparability—consult the intersection-only sensitivity table.
* **Multiple cases:** Use the exploratory q-values to control for multiple comparisons before declaring broad improvements.